In [1]:
import sys
import os

# Set the path to your Google Cloud Service Account JSON key

# Proje kök dizinini Python yoluna ekle
project_root = os.getcwd()
if not os.path.isdir(os.path.join(project_root, "fin_agent")):
	project_root = os.path.abspath(os.path.join(project_root, ".."))

if project_root not in sys.path:
	sys.path.append(project_root)

In [2]:
from dotenv import load_dotenv
import vertexai
import os

# Notebook test klasöründe olduğu için env dosyasını proje kökünden okuyoruz
load_dotenv(os.path.join(project_root, ".env"), override=True)

gcp_key = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
gcp_project = os.getenv("GOOGLE_CLOUD_PROJECT")
gcp_location = os.getenv("GOOGLE_CLOUD_LOCATION", "us-central1")

# gcp_key stringi ("gcp-key.json") dönüyordu ve mutlak path (absolute path) değildi.
# Eğer dosyayı proje klasörü altından okuyacaksa path'i proje köküyle birleştirmeliyiz.
if gcp_key and not os.path.isabs(gcp_key):
    gcp_key = os.path.join(project_root, gcp_key)

os.environ["GOOGLE_CLOUD_PROJECT"] = gcp_project
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = gcp_key

# Yetkinlendirmeyi doğru dosyayla başlatıyoruz
vertexai.init(project=gcp_project, location=gcp_location)

from fin_agent.utils.nodes import orchestrator, financialAgent, marketAgent
from fin_agent.utils.state import AgentState

/home/berk/finagent-systems/.venv/lib/python3.12/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [3]:
# 1. Başlangıç State'ini tasarlayalım
initial_state: AgentState = {
    "company_name": "TÜRK HAVA YOLLARI A.O.", # Test etmek istediğiniz şirketi buradan değiştirebilirsiniz
    "instructions": [],
    "messages": [],
    "financial_kpis": [],
    "market_sentiment": [],
    "audit_log": [],
    "loop_step": 0,
    "final_report": "",
    "credit_decision": "PENDING",
    "next_node": ""
}

print("Initial state başarıyla oluşturuldu. Hedef şirket:", initial_state["company_name"])

Initial state başarıyla oluşturuldu. Hedef şirket: TÜRK HAVA YOLLARI A.O.


In [4]:
# 2. Orchestrator Node'unu Test Edelim
print("Orchestrator çalıştırılıyor...")
orchestrator_result = orchestrator(initial_state)

print("\n--- ORCHESTRATOR ÇIKTISI (INSTRUCTIONS) ---")
for instruction in orchestrator_result.get("instructions", []):
    print(instruction.content)

# Bir sonraki adıma (Financial Agent) geçmeden önce state'i güncelleyelim.
# LangGraph akışı içinde state güncellemeleri otomatik yapılır ancak biz manuel olarak state birleştiriyoruz.
test_state = initial_state.copy()
test_state["instructions"] = orchestrator_result.get("instructions", [])

Orchestrator çalıştırılıyor...

--- ORCHESTRATOR ÇIKTISI (INSTRUCTIONS) ---
--- ORCHESTRATOR ANALIZ PLANI ---
Strateji: Hata: JSON parse edilemedi. Manuel planlama devreye alınıyor.
Gerekçe: None


In [8]:
# 3. Financial Agent'ı Test Edelim
import json

# UYARI: `financialAgent` içerisinde `pdf_path = "./sample_statement.pdf"` hardcode edilmiştir.
# Bu kodun başarılı çalışması için projenizin olduğu dizinde (veya notebook çalışma dizininde) 
# "sample_statement.pdf" isminde gerçek veya mock bir finansal pdf dosyası olmak zorundadır.

print("Financial Agent çalıştırılıyor... (PDF okuma ve Gemini Vision isteği gönderiliyor)")
financial_result = financialAgent(test_state)

print("\n--- FINANCIAL AGENT MESAJI ---")
print(financial_result.get("messages", [None])[0].content)

print("\n--- ÇIKARILAN FİNANSAL METRİKLER (financial_kpis) ---")
# Çıktıyı düzgün okuyabilmek için JSON formatında güzelce yazdıralım
print(json.dumps(financial_result.get("financial_kpis", []), indent=2, ensure_ascii=False))

Financial Agent çalıştırılıyor... (PDF okuma ve Gemini Vision isteği gönderiliyor)

--- FINANCIAL AGENT MESAJI ---
Finansal analiz tamamlandı, KPI'lar çıkarıldı.

--- ÇIKARILAN FİNANSAL METRİKLER (financial_kpis) ---
[
  {
    "company_info": {
      "company_name": "TÜRK HAVA YOLLARI A.O.",
      "period": "1 Ocak - 31 Aralık 2025"
    },
    "liquidity_metrics": {
      "current_ratio": 0.987189446860085,
      "quick_ratio": null
    },
    "leverage_and_debt": {
      "total_assets": 1996745.0,
      "total_debt": null,
      "total_equity": 911256.0,
      "debt_to_equity": null,
      "interest_coverage_ratio": null
    },
    "profitability_metrics": {
      "revenue": 936472.0,
      "gross_margin": 0.166110034608304,
      "ebitda": null,
      "ebitda_margin": null,
      "net_profit": 118117.0
    },
    "cash_flow_metrics": {
      "operating_cash_flow": null,
      "free_cash_flow": null
    }
  }
]


In [6]:
# 4. Market Agent'ı Test Edelim
import json

print("Market Agent çalıştırılıyor... (Tavily üzerinden internet taraması ve Gemini Analizi başlatıldı)")

# Burada test_state içerisinde daha önce orchestrator'un ürettiği instructions ve company_name bulunuyor.
market_result = marketAgent(test_state)

print("\n--- MARKET AGENT MESAJI ---")
print(market_result.get("messages", [None])[0].content)

print("\n--- ÇIKARILAN PİYASA RİSK METRİKLERİ (market_sentiment) ---")
# Çıktıyı JSON formatında inceleyelim
print(json.dumps(market_result.get("market_sentiment", []), indent=2, ensure_ascii=False))

Market Agent çalıştırılıyor... (Tavily üzerinden internet taraması ve Gemini Analizi başlatıldı)

--- MARKET AGENT MESAJI ---
Piyasa, haber akışı ve sektörel risk analizi tamamlandı.

--- ÇIKARILAN PİYASA RİSK METRİKLERİ (market_sentiment) ---
[
  {
    "market_analysis": {
      "company_name": "TÜRK HAVA YOLLARI A.O.",
      "sector_risk_score": 72,
      "sentiment": "NEGATIVE",
      "key_risks": [
        "2024 yılında karın %43 düşmesi ile azalan karlılık",
        "Sürekli enflasyonist baskılar ve değişken jet yakıtı fiyatları",
        "Türkiye'deki siyasi istikrarsızlık ve kur oynaklığı",
        "Avrupa'da potansiyel talep yavaşlaması ve artan korumacılık önlemleri"
      ],
      "competitor_analysis": "Türk Hava Yolları, Emirates, Qatar Airways, Lufthansa ve Air France-KLM gibi küresel tam hizmet taşıyıcıları ile çevik düşük maliyetli rakiplerle rekabet etmektedir. İstanbul Havalimanı'ndaki benzersiz aktarma merkezi modeli, coğrafi konumu ve göreceli olarak istikrarlı kargo